# Notebook 02 v3 - Pump Digital Twin (Telemetry + Labels Separation)


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm


In [ ]:

class PumpTwin:

    def __init__(self, asset_id, failure_scenario):

        self.asset_id = asset_id
        self.failure_scenario = failure_scenario

        self.health = 100.0
        self.runtime_hours = 0

        self.bearing_wear = 0
        self.seal_wear = 0
        self.cavitation_index = 0

    def update_health(self):

        wear = np.random.uniform(0.00015, 0.0008)

        if self.failure_scenario == "Bearing Failure":
            self.bearing_wear += wear * 5
            self.health -= wear * 1.8

        elif self.failure_scenario == "Seal Leakage":
            self.seal_wear += wear * 5
            self.health -= wear * 1.5

        elif self.failure_scenario == "Cavitation":
            self.cavitation_index += wear * 5
            self.health -= wear * 1.4

        else:
            self.health -= wear * 0.3

        self.health = max(self.health, 0)
        self.runtime_hours += 1/60

    def get_rul(self):
        return round((self.health / 100) * 180, 2)

    def get_failure_next_30_days(self):

        if self.get_rul() <= 30:
            return 1

        return 0

    def generate_telemetry(self):

        self.update_health()

        bearing_temp = (
            55 +
            self.bearing_wear * 8 +
            np.random.normal(0,1)
        )

        motor_temp = (
            60 +
            (100-self.health)*0.4 +
            np.random.normal(0,1)
        )

        vibration = (
            1.2 +
            self.bearing_wear*1.2 +
            self.cavitation_index*1.8 +
            np.random.normal(0,0.05)
        )

        flow_rate = (
            300 -
            self.seal_wear*6 -
            self.cavitation_index*3 +
            np.random.normal(0,2)
        )

        suction_pressure = (
            3 -
            self.cavitation_index*0.2 +
            np.random.normal(0,0.05)
        )

        discharge_pressure = (
            10 -
            self.seal_wear*0.3 +
            np.random.normal(0,0.05)
        )

        motor_current = (
            25 +
            self.bearing_wear*3 +
            np.random.normal(0,0.2)
        )

        lubrication_level = max(
            20,
            100-self.bearing_wear*2
        )

        seal_leakage_rate = self.seal_wear * 5

        telemetry = {

            "asset_id":self.asset_id,

            "bearing_temp":round(bearing_temp,2),
            "motor_temp":round(motor_temp,2),
            "vibration":round(vibration,2),
            "flow_rate":round(flow_rate,2),
            "suction_pressure":round(suction_pressure,2),
            "discharge_pressure":round(discharge_pressure,2),
            "motor_current":round(motor_current,2),
            "lubrication_level":round(lubrication_level,2),
            "seal_leakage_rate":round(seal_leakage_rate,2)
        }

        labels = {

            "asset_id":self.asset_id,

            "failure_mode":self.failure_scenario,

            "rul_days":self.get_rul(),

            "failure_next_30_days":
                self.get_failure_next_30_days()
        }

        return telemetry, labels


In [ ]:

pumps = [

    PumpTwin("P-101","Bearing Failure"),
    PumpTwin("P-102","Seal Leakage"),
    PumpTwin("P-103","Cavitation"),
    PumpTwin("P-104","Bearing Failure"),
    PumpTwin("P-105","Healthy")
]


In [ ]:

telemetry_records = []
label_records = []

minutes = 90 * 24 * 60

start_time = datetime.now()

for minute in tqdm(range(minutes)):

    current_time = (
        start_time +
        timedelta(minutes=minute)
    )

    for pump in pumps:

        telemetry, labels = (
            pump.generate_telemetry()
        )

        telemetry["timestamp"] = current_time
        labels["timestamp"] = current_time

        telemetry_records.append(telemetry)
        label_records.append(labels)

telemetry_df = pd.DataFrame(telemetry_records)
labels_df = pd.DataFrame(label_records)

print("Telemetry:", telemetry_df.shape)
print("Labels:", labels_df.shape)

telemetry_df.to_csv(
    "pump_telemetry.csv",
    index=False
)

labels_df.to_csv(
    "pump_labels.csv",
    index=False
)

telemetry_df.head()
